# MCP — Model Context Protocol

## Northstar: safe deployment context for an incident agent

An incident agent needs release evidence for deployment `842`. An MCP client connects through an enterprise gateway to a deployment MCP server. We will negotiate eligible capabilities, invoke a typed read tool, block a write, and map MCP into the wider agent interoperability stack.

**Outcomes:** understand MCP architecture; distinguish tools/resources/prompts; apply capability negotiation, authentication, authorization, gateways, and security controls; and connect a bounded agent without granting protocol content authority. Default examples are deterministic and credential-free.

![MCP architecture](../../../assets/mcp-architecture.svg)

The agent host owns model behavior and policy. The MCP client/gateway filters and traces capabilities. Servers expose tools, resources, and prompts. Identity, delegated scopes, tenant policy, validation, approvals, budgets, audit, and revocation remain application-owned.

## Step 1 — discover a least-privilege capability surface

MCP exists to standardize client/server integrations around context and operations. Negotiation says what protocol features and capabilities are available; it does not decide what the agent is authorized to do. An enterprise host should request or present only the capabilities the current delegated scopes permit. This reduces both model confusion and excessive agency.

In [1]:
from pathlib import Path
import sys
TOPIC = Path.cwd()
if not (TOPIC / 'lab.py').exists():
    TOPIC = Path.cwd() / 'curriculum' / 'advanced' / '13-mcp-model-context-protocol'
sys.path.insert(0, str(TOPIC))
from lab import invoke, negotiate

visible = negotiate({'deployments.read', 'incidents.read'})
for capability in visible:
    print(capability)
assert 'rollback_deployment' not in {c.name for c in visible}

Capability(name='deployment://842', kind='resource', scopes=frozenset({'deployments.read'}), side_effect=False)
Capability(name='investigate-release', kind='prompt', scopes=frozenset({'incidents.read'}), side_effect=False)
Capability(name='get_deployment', kind='tool', scopes=frozenset({'deployments.read'}), side_effect=False)


## Step 2 — distinguish tools, resources, and prompts

A **tool** is a typed operation; it may have side effects. A **resource** is contextual data, often addressable by a URI. A **prompt** is a reusable server-provided template. All three are untrusted inputs to the host: a tool description or resource can be poisoned, and a prompt cannot override system instructions, policy, authorization, or budgets.

The read tool below accepts exactly one deployment ID. In real systems, validate the JSON schema, tenant predicate, data classification, result provenance/freshness, rate limit, and trace. Never let a model-generated string become an unvalidated production call.

In [2]:
result = invoke('get_deployment', {'deployment_id': '842'}, {'deployments.read'})
print(result)
assert result['trusted_as'] == 'untrusted-data'

try:
    invoke('get_deployment', {'deployment_id': 'other-tenant'}, {'deployments.read'})
except ValueError as error:
    print('schema blocked:', error)

{'deployment_id': '842', 'version': '2026.08.10', 'trusted_as': 'untrusted-data'}
schema blocked: strict tool schema rejected arguments


## Step 3 — authentication, authorization, remote MCP, and gateways

Authentication establishes the client/principal; authorization filters each resource/tool/action for the delegated user/agent scope. Remote MCP adds transport, server provenance, availability, network, egress, secret, and tenant concerns. An enterprise gateway should provide trusted registration, server/version allowlisting, token exchange, per-tenant routing, capability filtering, argument/result validation, action approval, budgeting, audit, observability, and emergency revocation.

Discovery is not approval. A previously visible capability is not perpetual authorization. A tool result is data, not an instruction.

In [3]:
try:
    invoke('rollback_deployment', {'idempotency_key': 'rollback-842'}, {'deployments.write'})
except PermissionError as error:
    print('approval gate:', error)

proposal = invoke('rollback_deployment', {'idempotency_key': 'rollback-842'}, {'deployments.write'}, approved=True)
print(proposal)
assert proposal['idempotency_key'] == 'rollback-842'

approval gate: side effect requires human-approved action fingerprint
{'status': 'proposal-submitted', 'idempotency_key': 'rollback-842'}


## Step 4 — MCP security failure cases

Test indirect prompt injection in resources/prompts, malicious tool metadata, capability overexposure, confused-deputy delegation, cross-tenant arguments, leaked credentials, remote-server compromise, side-effect replay, and stale results. Defenses are layered: server review/registry/provenance; authentication and short-lived scopes; authorization-aware catalogues; typed validation; content isolation; human approval; idempotency/reconciliation; egress/budgets; traces/audit; and revocation. A single prompt warning is not a security architecture.

## Step 5 — MCP is one layer of interoperability

Use MCP for tools/context. Use A2A for remote agent discovery, task lifecycle, delegation, and collaboration. Use AG-UI/A2UI for agent/user interactions and safe dynamic UI. Use commerce/payment adapters for UCP/AP2-style flows behind distinct consent, merchant, provider, fraud, and financial controls. The emerging ecosystem is compositional: every protocol hop must retain identity, intent, tenant, scope, policy, traces, and revocation rather than treating interoperability as automatic trust.

**Exercises:** implement per-tenant resource filtering; simulate a poisoned prompt; add a remote-server timeout and retry only a safe read; record redacted trace events; create a server registry review record; and prove a stale approval cannot replay a rollback.

References: [MCP specification](https://modelcontextprotocol.io/specification/), [MCP authorization](https://modelcontextprotocol.io/specification/2025-03-26/basic/authorization), [A2A](https://a2a-protocol.org/latest/), [interoperability survey](https://arxiv.org/abs/2505.02279), [OWASP agentic applications](https://genai.owasp.org/resource/owasp-top-10-for-agentic-applications/).